# BÀI TOÁN VẬN TẢI — CHƯƠNG 3
### Bộ giải tổng hợp bằng Python (thuật toán thế vị) — chạy trên Google Colab

Notebook này giải **mọi dạng bài toán vận tải trong Chương 3** bằng một bộ giải duy nhất:
phương án xuất phát theo *ưu tiên cước phí nhỏ nhất*, tối ưu hoá bằng *thuật toán thế vị*,
tự xử lý **không cân bằng thu–phát** (lập trạm giả) và **ô cấm** (cước phí M lớn).
Mỗi lần giải đều **in đầy đủ kết quả trung gian của từng bước**.

## Cách dùng
1. Chạy **Mục 1 — Bộ giải** một lần (Runtime ▸ Run all, hoặc chạy ô code Mục 1).
2. Chạy ô code ở bất kỳ ví dụ/bài tập nào bên dưới để xem lời giải chi tiết.
3. Muốn giải bài mới: dùng **Mục 8 — Mẫu**, sửa 4 dữ liệu `supply, demand, cost, forbidden`.

> Mẹo: mở **Table of contents** ở thanh bên trái Colab (biểu tượng ☰) để nhảy nhanh giữa các mục.

## Mục lục
- [1. Bộ giải tổng quát (chạy trước)](#engine)
- [2. Ví dụ 3.27 — thuật toán thế vị (cân bằng) — f = 800](#vd327)
- [3. Bài tập 8 — cân bằng — f = 3550](#bai8)
- [4. Bài tập 2 — không cân bằng thu–phát (mục 3.6.1) — f = 3200](#bai2)
- [5. Ví dụ 3.32 — ô cấm (mục 3.6.2, cước phí M) — f = 610](#vd332)
- [6. Bài tập 5 — ô cấm — f = 1510](#bai5)
- [7. Ghi chú phương pháp](#ghichu)
- [8. Mẫu: giải bài toán của bạn](#mau)

> 📝 Sau khi dùng thử, xin góp ý qua [Phiếu đánh giá sản phẩm](https://docs.google.com/forms/d/e/1FAIpQLScNqxvL5WIb8PxsOMvBbOrmmGTwV56bMyRV8-Nb_2Zvj225iw/viewform) (khoảng 5 phút).

<a name="engine"></a>

## 1. Bộ giải tổng quát  *(chạy ô này trước tiên)*

Ô code dưới định nghĩa hàm **`giai_bai_toan(supply, demand, cost, forbidden=[], title="")`**.
Chỉ cần chạy một lần cho cả phiên làm việc.

In [1]:
# -*- coding: utf-8 -*-
"""
BỘ GIẢI TỔNG QUÁT BÀI TOÁN VẬN TẢI - CHƯƠNG 3
Xử lý được: cân bằng / không cân bằng thu-phát (tự lập trạm giả) / có ô cấm (cước phí M lớn).
Phương pháp: ưu tiên cước phí nhỏ nhất (PA xuất phát) + thuật toán THẾ VỊ.
Gọi:  giai_bai_toan(supply, demand, cost, forbidden=[], title="...")
"""
import numpy as np
import pandas as pd
pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)


# ----- Số học M lớn: Big(p, q) = p*M + q -----
class Big:
    __slots__ = ("p", "q")
    def __init__(self, p, q): self.p, self.q = p, q
    def __add__(s, o): return Big(s.p + o.p, s.q + o.q)
    def __sub__(s, o): return Big(s.p - o.p, s.q - o.q)
    def __neg__(s): return Big(-s.p, -s.q)
    def scaled(s, k): return Big(s.p * k, s.q * k)
    def key(s): return (s.p, s.q)
    def is_neg(s): return s.key() < (0, 0)
    def __str__(s):
        p, q = int(s.p), int(s.q)
        if p == 0: return f"{q}"
        base = "M" if p == 1 else ("-M" if p == -1 else f"{p}M")
        return base if q == 0 else base + (f"+{q}" if q > 0 else f"{q}")


# ----- biến toàn cục dùng chung -----
G = {}   # gói trạng thái để tránh rối biến toàn cục


def _show_alloc(alloc, basis, title=""):
    M, N, cost, ROWS, COLS = G["M"], G["N"], G["cost"], G["ROWS"], G["COLS"]
    disp = np.full((M, N), "", dtype=object)
    for i in range(M):
        for j in range(N):
            tag = f"(c={cost[i][j]})"
            disp[i, j] = f"{int(round(alloc[i,j]))} {tag}" if (i, j) in basis else f". {tag}"
    df = pd.DataFrame(disp, index=[f"{ROWS[i]}={G['supply'][i]}" for i in range(M)],
                      columns=[f"{COLS[j]}={G['demand'][j]}" for j in range(N)])
    if title: print(title)
    print(df.to_string()); print()


def _show_reduced(delta, u, v, basis, title=""):
    M, N, ROWS, COLS = G["M"], G["N"], G["ROWS"], G["COLS"]
    disp = np.empty((M, N), dtype=object)
    for i in range(M):
        for j in range(N):
            disp[i, j] = "[0]" if (i, j) in basis else str(delta[i][j])
    df = pd.DataFrame(disp, index=[f"{ROWS[i]} (u={u[i]})" for i in range(M)],
                      columns=[f"{COLS[j]} (v={v[j]})" for j in range(N)])
    if title: print(title)
    print(df.to_string()); print()


def _obj(alloc, basis):
    tot = Big(0, 0)
    for (i, j) in basis:
        tot = tot + G["cost"][i][j].scaled(alloc[i, j])
    return tot


def _setup(supply0, demand0, cost0, forbidden):
    sa, sb = sum(supply0), sum(demand0)
    oM, oN = len(supply0), len(demand0)
    fset = set(forbidden)
    BIGM = Big(1, 0)
    print("=" * 72)
    print("BƯỚC 1 - CHUẨN BỊ: đặt ô cấm = M, kiểm tra cân bằng thu-phát")
    print("=" * 72)
    print("Ô cấm:", ", ".join(f"({r+1},{c+1})" for r, c in forbidden) if forbidden else "(không có)")
    cost = [[BIGM if (i, j) in fset else Big(0, cost0[i][j]) for j in range(oN)] for i in range(oM)]
    supply, demand = list(supply0), list(demand0)
    print(f"Tổng phát Sum(a) = {sa};  Tổng thu Sum(b) = {sb}")
    fake = None
    if sa == sb:
        print("=> Cân bằng thu-phát.\n")
    elif sa < sb:
        d = sb - sa
        print(f"=> Sum(a) < Sum(b): thiếu {d}. Lập TRẠM PHÁT GIẢ (dòng {oM+1}), cước phí 0.\n")
        cost.append([Big(0, 0) for _ in range(oN)]); supply.append(d); fake = "row"
    else:
        d = sa - sb
        print(f"=> Sum(a) > Sum(b): thừa {d}. Lập TRẠM THU GIẢ (cột {oN+1}), cước phí 0.\n")
        for i in range(oM): cost[i].append(Big(0, 0))
        demand.append(d); fake = "col"
    M, N = len(supply), len(demand)
    ROWS = [f"A{i+1}" for i in range(M)]
    COLS = [f"B{j+1}" for j in range(N)]
    if fake == "row": ROWS[-1] += "*"
    if fake == "col": COLS[-1] += "*"
    G.update(cost=cost, supply=supply, demand=demand, M=M, N=N, ROWS=ROWS, COLS=COLS,
             fake=fake, orig_M=oM, orig_N=oN, forb=fset,
             supply0=supply0, demand0=demand0)
    print("Bảng cước phí (M = ô cấm; * = trạm giả):")
    print(pd.DataFrame([[str(cost[i][j]) for j in range(N)] for i in range(M)],
                       index=[f"{ROWS[i]}={supply[i]}" for i in range(M)],
                       columns=[f"{COLS[j]}={demand[j]}" for j in range(N)]).to_string())
    print()


def _min_cost_start():
    M, N, cost = G["M"], G["N"], G["cost"]
    a = np.array(G["supply"], float); b = np.array(G["demand"], float)
    alloc = np.zeros((M, N)); basis = set()
    row_ok = [True] * M; col_ok = [True] * N
    print("=" * 72)
    print("BƯỚC 2 - PHƯƠNG ÁN XUẤT PHÁT (ưu tiên cước phí nhỏ nhất)")
    print("=" * 72)
    step = 0
    while any(row_ok) and any(col_ok):
        best = None
        for i in range(M):
            if not row_ok[i]: continue
            for j in range(N):
                if col_ok[j] and (best is None or cost[i][j].key() < cost[best[0]][best[1]].key()):
                    best = (i, j)
        i, j = best; q = min(a[i], b[j])
        alloc[i, j] = q; basis.add((i, j)); a[i] -= q; b[j] -= q; step += 1
        print(f"Bước {step}: ô ({i+1},{j+1}) c={cost[i][j]} -> x[{i+1},{j+1}] = {int(q)}")
        if a[i] == 0: row_ok[i] = False
        if b[j] == 0: col_ok[j] = False
    print()
    _show_alloc(alloc, basis, ">> Phương án xuất phát:")
    print(f">> Số ô chọn = {len(basis)};  m + n - 1 = {M + N - 1}")
    print(f">> f(xuất phát) = {_obj(alloc, basis)}\n")
    return alloc, basis


def _components(basis):
    M, N = G["M"], G["N"]; parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(M): parent[("R", i)] = ("R", i)
    for j in range(N): parent[("C", j)] = ("C", j)
    for (i, j) in basis: parent[find(("R", i))] = find(("C", j))
    return find


def _repair(alloc, basis):
    M, N, cost, fset = G["M"], G["N"], G["cost"], G["forb"]
    if len(basis) >= M + N - 1: return alloc, basis
    print(f"XỬ LÝ SUY BIẾN: bổ sung {M + N - 1 - len(basis)} ô chọn 0 (tránh ô cấm).")
    while len(basis) < M + N - 1:
        find = _components(basis)
        cands = [(i, j) for i in range(M) for j in range(N)
                 if (i, j) not in basis and (i, j) not in fset and find(("R", i)) != find(("C", j))]
        if not cands:
            cands = [(i, j) for i in range(M) for j in range(N)
                     if (i, j) not in basis and find(("R", i)) != find(("C", j))]
        add = min(cands, key=lambda c: cost[c[0]][c[1]].key())
        basis.add(add); print(f"  + thêm ô chọn 0 tại ({add[0]+1},{add[1]+1})")
    print()
    _show_alloc(alloc, basis, ">> Sau khi bổ sung ô chọn 0:")
    return alloc, basis


def _potentials(basis):
    M, N, cost = G["M"], G["N"], G["cost"]
    u = [None] * M; v = [None] * N; u[0] = Big(0, 0)
    adj = {}
    for (i, j) in basis:
        adj.setdefault(("R", i), []).append(("C", j)); adj.setdefault(("C", j), []).append(("R", i))
    stack = [("R", 0)]; seen = {("R", 0)}
    while stack:
        t, idx = stack.pop()
        for nb in adj.get((t, idx), []):
            if nb in seen: continue
            seen.add(nb); tt, ii = nb
            if t == "R": v[ii] = -cost[idx][ii] - u[idx]
            else: u[ii] = -cost[ii][idx] - v[idx]
            stack.append(nb)
    u = [x if x is not None else Big(0, 0) for x in u]
    v = [x if x is not None else Big(0, 0) for x in v]
    return u, v


def _reduced(u, v):
    M, N, cost = G["M"], G["N"], G["cost"]
    return [[cost[i][j] + u[i] + v[j] for j in range(N)] for i in range(M)]


def _find_cycle(basis, enter):
    i0, j0 = enter; adj = {}
    for (i, j) in basis:
        adj.setdefault(("R", i), []).append(("C", j)); adj.setdefault(("C", j), []).append(("R", i))
    prev = {("R", i0): None}; stack = [("R", i0)]
    while stack:
        node = stack.pop()
        if node == ("C", j0): break
        for nb in adj.get(node, []):
            if nb not in prev: prev[nb] = node; stack.append(nb)
    path = []; node = ("C", j0)
    while node is not None: path.append(node); node = prev[node]
    path.reverse()
    cells = [enter]
    for aa, bb in zip(path, path[1:]):
        (ta, ia), (tb, ib) = aa, bb
        cells.append((ia, ib) if ta == "R" else (ib, ia))
    return cells


def giai_bai_toan(supply, demand, cost, forbidden=None, title=""):
    """Giải một bài toán vận tải và in đầy đủ các bước trung gian."""
    forbidden = forbidden or []
    if title:
        print("#" * 72); print("#  " + title); print("#" * 72)
    _setup(supply, demand, cost, forbidden)
    alloc, basis = _min_cost_start()
    alloc, basis = _repair(alloc, basis)
    M, N = G["M"], G["N"]
    it = 0
    while True:
        it += 1
        print("=" * 72); print(f"VÒNG LẶP {it}"); print("=" * 72)
        u, v = _potentials(basis); delta = _reduced(u, v)
        print("BƯỚC 3 - Quy 0 ô chọn: thế vị u_i, v_j và cước phí mới c'_ij")
        _show_reduced(delta, u, v, basis)
        nb = [(i, j) for i in range(M) for j in range(N) if (i, j) not in basis]
        cell = min(nb, key=lambda c: delta[c[0]][c[1]].key())
        if not delta[cell[0]][cell[1]].is_neg():
            print("BƯỚC 4 - Mọi c'_ij >= 0  =>  ĐÃ TỐI ƯU. Dừng.\n"); break
        print(f"BƯỚC 4 - Nhỏ nhất c'[{cell[0]+1},{cell[1]+1}] = {delta[cell[0]][cell[1]]} < 0  =>  chưa tối ưu.\n")
        cyc = _find_cycle(basis, cell)
        signs = ["+" if k % 2 == 0 else "-" for k in range(len(cyc))]
        minus = [cyc[k] for k in range(len(cyc)) if signs[k] == "-"]
        q = min(alloc[c] for c in minus); leave = min(minus, key=lambda c: alloc[c])
        print(f"BƯỚC 5 - Đưa ô ({cell[0]+1},{cell[1]+1}) vào. Chu trình: " +
              " -> ".join(f"({c[0]+1},{c[1]+1}){s}" for c, s in zip(cyc, signs)))
        print(f"         q = {int(q)};  ô ({leave[0]+1},{leave[1]+1}) thành ô loại." +
              ("  (suy biến: q=0)" if q == 0 else ""))
        for c, s in zip(cyc, signs): alloc[c] += q if s == "+" else -q
        basis.add(cell); basis.discard(leave)
        _show_alloc(alloc, basis, ">> Phương án mới:")
        print(f">> f = {_obj(alloc, basis)}\n")

    # kết quả
    Xbar = np.zeros((M, N))
    for (i, j) in basis: Xbar[i, j] = alloc[i, j]
    ROWS, COLS = G["ROWS"], G["COLS"]
    print("=" * 72); print("KẾT QUẢ"); print("=" * 72)
    print("Phương án tối ưu (bảng đã mở rộng M / trạm giả):")
    print(pd.DataFrame(Xbar.astype(int), index=[f"{ROWS[i]}={G['supply'][i]}" for i in range(M)],
                       columns=[f"{COLS[j]}={G['demand'][j]}" for j in range(N)]).to_string()); print()
    fset = G["forb"]
    used = [(i, j) for (i, j) in basis if (i, j) in fset and alloc[i, j] > 1e-9]
    if used:
        print("!! Ô cấm là ô chọn với lượng > 0:", ", ".join(f"({i+1},{j+1})={int(alloc[i,j])}" for i, j in used))
        print("!! => BÀI TOÁN GỐC VÔ NGHIỆM.\n"); return None, None
    if fset: print("Kiểm tra ô cấm: mọi ô cấm đều x = 0 => bài toán gốc CÓ nghiệm.")
    oM, oN, fake = G["orig_M"], G["orig_N"], G["fake"]
    if fake == "row":
        X = Xbar[:oM, :]; print(f"Cắt bỏ DÒNG GIẢ (A{M}). ", end="")
    elif fake == "col":
        X = Xbar[:, :oN]; print(f"Cắt bỏ CỘT GIẢ (B{N}). ", end="")
    else:
        X = Xbar
    print("Phương án tối ưu của BÀI TOÁN GỐC:")
    print(pd.DataFrame(X.astype(int), index=[f"A{i+1}={G['supply0'][i]}" for i in range(oM)],
                       columns=[f"B{j+1}={G['demand0'][j]}" for j in range(oN)]).to_string())
    fmin = _obj(alloc, basis)
    print(f"\n>>> f_min = {fmin}\n")
    # diễn giải trạm giả
    if fake == "row":
        print("Ý nghĩa (thiếu hàng): lượng trên dòng giả = phần CẦU không được đáp ứng.")
        for j in range(oN):
            if int(Xbar[M-1, j]) > 0: print(f"   - B{j+1} thiếu {int(Xbar[M-1,j])} đơn vị.")
    elif fake == "col":
        print("Ý nghĩa (thừa hàng): lượng trên cột giả = phần hàng bị TỒN (không phát đi).")
        for i in range(oM):
            if int(Xbar[i, N-1]) > 0: print(f"   - A{i+1} tồn {int(Xbar[i,N-1])} đơn vị.")
    print()
    return X, fmin

<a name="vd327"></a>

## 2. Ví dụ 3.27 — thuật toán thế vị (bài cân bằng)

Bài toán 4×5 cân bằng (Σa = Σb = 150). Kết quả: **f_min = 800**.

In [2]:
giai_bai_toan(
    supply=[30, 20, 40, 60],
    demand=[30, 20, 25, 35, 40],
    cost=[[13,  7,  6,  2, 12],
          [ 5,  1, 10,  5, 11],
          [10,  5,  3,  7, 14],
          [ 6,  3,  2, 11, 10]],
    title="Ví dụ 3.27 - thuat toan the vi")

########################################################################
#  Ví dụ 3.27 - thuat toan the vi
########################################################################
BƯỚC 1 - CHUẨN BỊ: đặt ô cấm = M, kiểm tra cân bằng thu-phát
Ô cấm: (không có)
Tổng phát Sum(a) = 150;  Tổng thu Sum(b) = 150
=> Cân bằng thu-phát.

Bảng cước phí (M = ô cấm; * = trạm giả):
      B1=30 B2=20 B3=25 B4=35 B5=40
A1=30    13     7     6     2    12
A2=20     5     1    10     5    11
A3=40    10     5     3     7    14
A4=60     6     3     2    11    10

BƯỚC 2 - PHƯƠNG ÁN XUẤT PHÁT (ưu tiên cước phí nhỏ nhất)
Bước 1: ô (2,2) c=1 -> x[2,2] = 20
Bước 2: ô (1,4) c=2 -> x[1,4] = 30
Bước 3: ô (4,3) c=2 -> x[4,3] = 25
Bước 4: ô (4,1) c=6 -> x[4,1] = 30
Bước 5: ô (3,4) c=7 -> x[3,4] = 5
Bước 6: ô (4,5) c=10 -> x[4,5] = 5
Bước 7: ô (3,5) c=14 -> x[3,5] = 35

>> Phương án xuất phát:
          B1=30     B2=20     B3=25     B4=35      B5=40
A1=30  . (c=13)   . (c=7)   . (c=6)  30 (c=2)   . (c=12)
A2=20   

(array([[ 0.,  0.,  0., 30.,  0.],
        [10., 10.,  0.,  0.,  0.],
        [ 0., 10., 25.,  5.,  0.],
        [20.,  0.,  0.,  0., 40.]]),
 <__main__.Big at 0x7f30fa339630>)

<a name="bai8"></a>

## 3. Bài tập 8 — bài cân bằng

Bài toán 4×5 cân bằng (Σa = Σb = 400). Kết quả: **f_min = 3550**.

In [3]:
giai_bai_toan(
    supply=[100, 80, 140, 80],
    demand=[120, 70, 30, 50, 130],
    cost=[[11,  9, 13, 11, 10],
          [ 6,  8,  5,  7,  9],
          [14, 13,  6, 15, 12],
          [10, 10, 16,  7,  8]],
    title="Bai 8")

########################################################################
#  Bai 8
########################################################################
BƯỚC 1 - CHUẨN BỊ: đặt ô cấm = M, kiểm tra cân bằng thu-phát
Ô cấm: (không có)
Tổng phát Sum(a) = 400;  Tổng thu Sum(b) = 400
=> Cân bằng thu-phát.

Bảng cước phí (M = ô cấm; * = trạm giả):
       B1=120 B2=70 B3=30 B4=50 B5=130
A1=100     11     9    13    11     10
A2=80       6     8     5     7      9
A3=140     14    13     6    15     12
A4=80      10    10    16     7      8

BƯỚC 2 - PHƯƠNG ÁN XUẤT PHÁT (ưu tiên cước phí nhỏ nhất)
Bước 1: ô (2,3) c=5 -> x[2,3] = 30
Bước 2: ô (2,1) c=6 -> x[2,1] = 50
Bước 3: ô (4,4) c=7 -> x[4,4] = 50
Bước 4: ô (4,5) c=8 -> x[4,5] = 30
Bước 5: ô (1,2) c=9 -> x[1,2] = 70
Bước 6: ô (1,5) c=10 -> x[1,5] = 30
Bước 7: ô (3,5) c=12 -> x[3,5] = 70
Bước 8: ô (3,1) c=14 -> x[3,1] = 70

>> Phương án xuất phát:
           B1=120     B2=70     B3=30     B4=50     B5=130
A1=100   . (c=11)  70 (c=9)  . (c=1

(array([[ 30.,  70.,   0.,   0.,   0.],
        [ 80.,   0.,   0.,   0.,   0.],
        [ 10.,   0.,  30.,   0., 100.],
        [  0.,   0.,   0.,  50.,  30.]]),
 <__main__.Big at 0x7f30fa33b670>)

<a name="bai2"></a>

## 4. Bài tập 2 — không cân bằng thu–phát (mục 3.6.1)

Σa = 200 < Σb = 250 → chương trình **tự lập trạm phát giả**. Kết quả: **f_min = 3200** (B3 thiếu 30, B4 thiếu 20).

In [4]:
giai_bai_toan(
    supply=[80, 70, 50],
    demand=[100, 50, 30, 70],
    cost=[[10, 20, 40, 30],
          [20, 30, 50, 60],
          [60, 20, 70, 10]],
    title="Bai 2 - khong can bang")

########################################################################
#  Bai 2 - khong can bang
########################################################################
BƯỚC 1 - CHUẨN BỊ: đặt ô cấm = M, kiểm tra cân bằng thu-phát
Ô cấm: (không có)
Tổng phát Sum(a) = 200;  Tổng thu Sum(b) = 250
=> Sum(a) < Sum(b): thiếu 50. Lập TRẠM PHÁT GIẢ (dòng 4), cước phí 0.

Bảng cước phí (M = ô cấm; * = trạm giả):
       B1=100 B2=50 B3=30 B4=70
A1=80      10    20    40    30
A2=70      20    30    50    60
A3=50      60    20    70    10
A4*=50      0     0     0     0

BƯỚC 2 - PHƯƠNG ÁN XUẤT PHÁT (ưu tiên cước phí nhỏ nhất)
Bước 1: ô (4,1) c=0 -> x[4,1] = 50
Bước 2: ô (1,1) c=10 -> x[1,1] = 50
Bước 3: ô (3,4) c=10 -> x[3,4] = 50
Bước 4: ô (1,2) c=20 -> x[1,2] = 30
Bước 5: ô (2,2) c=30 -> x[2,2] = 20
Bước 6: ô (2,3) c=50 -> x[2,3] = 30
Bước 7: ô (2,4) c=60 -> x[2,4] = 20

>> Phương án xuất phát:
           B1=100      B2=50      B3=30      B4=70
A1=80   50 (c=10)  30 (c=20)   . (c=40)   . (

(array([[80.,  0.,  0.,  0.],
        [20., 50.,  0.,  0.],
        [ 0.,  0.,  0., 50.]]),
 <__main__.Big at 0x7f30fa33b790>)

<a name="vd332"></a>

## 5. Ví dụ 3.32 — bài toán có ô cấm (mục 3.6.2)

Ô cấm (3,2) và (3,4) → đặt cước phí **M** lớn. Kết quả: **f_min = 610**, ô cấm đều bằng 0.

In [5]:
giai_bai_toan(
    supply=[80, 70, 80, 60],
    demand=[40, 100, 60, 90],
    cost=[[1, 2, 4, 3],
          [2, 4, 5, 1],
          [4, 3, 2, 5],
          [2, 3, 1, 5]],
    forbidden=[(2, 1), (2, 3)],   # o cam (3,2) va (3,4)  -- chi so 0-based
    title="Vi du 3.32 - o cam")

########################################################################
#  Vi du 3.32 - o cam
########################################################################
BƯỚC 1 - CHUẨN BỊ: đặt ô cấm = M, kiểm tra cân bằng thu-phát
Ô cấm: (3,2), (3,4)
Tổng phát Sum(a) = 290;  Tổng thu Sum(b) = 290
=> Cân bằng thu-phát.

Bảng cước phí (M = ô cấm; * = trạm giả):
      B1=40 B2=100 B3=60 B4=90
A1=80     1      2     4     3
A2=70     2      4     5     1
A3=80     4      M     2     M
A4=60     2      3     1     5

BƯỚC 2 - PHƯƠNG ÁN XUẤT PHÁT (ưu tiên cước phí nhỏ nhất)
Bước 1: ô (1,1) c=1 -> x[1,1] = 40
Bước 2: ô (2,4) c=1 -> x[2,4] = 70
Bước 3: ô (4,3) c=1 -> x[4,3] = 60
Bước 4: ô (1,2) c=2 -> x[1,2] = 40
Bước 5: ô (3,2) c=M -> x[3,2] = 60
Bước 6: ô (3,4) c=M -> x[3,4] = 20

>> Phương án xuất phát:
          B1=40    B2=100     B3=60     B4=90
A1=80  40 (c=1)  40 (c=2)   . (c=4)   . (c=3)
A2=70   . (c=2)   . (c=4)   . (c=5)  70 (c=1)
A3=80   . (c=4)  60 (c=M)   . (c=2)  20 (c=M)
A4=60   

(array([[ 0., 60.,  0., 20.],
        [ 0.,  0.,  0., 70.],
        [20.,  0., 60.,  0.],
        [20., 40.,  0.,  0.]]),
 <__main__.Big at 0x7f30fa3385b0>)

<a name="bai5"></a>

## 6. Bài tập 5 — bài toán có ô cấm

Ô cấm (1,5) → cước phí **M**. Bài 4×5 cân bằng. Kết quả: **f_min = 1510**.

In [6]:
giai_bai_toan(
    supply=[110, 100, 60, 100],
    demand=[95, 80, 65, 35, 95],
    cost=[[ 6,  5,  2,  7,  0],
          [12,  5,  4,  7,  1],
          [ 9, 11,  5,  9,  9],
          [ 7,  4, 10,  6, 12]],
    forbidden=[(0, 4)],           # o cam (1,5)  -- chi so 0-based
    title="Bai 5 - o cam")

########################################################################
#  Bai 5 - o cam
########################################################################
BƯỚC 1 - CHUẨN BỊ: đặt ô cấm = M, kiểm tra cân bằng thu-phát
Ô cấm: (1,5)
Tổng phát Sum(a) = 370;  Tổng thu Sum(b) = 370
=> Cân bằng thu-phát.

Bảng cước phí (M = ô cấm; * = trạm giả):
       B1=95 B2=80 B3=65 B4=35 B5=95
A1=110     6     5     2     7     M
A2=100    12     5     4     7     1
A3=60      9    11     5     9     9
A4=100     7     4    10     6    12

BƯỚC 2 - PHƯƠNG ÁN XUẤT PHÁT (ưu tiên cước phí nhỏ nhất)
Bước 1: ô (2,5) c=1 -> x[2,5] = 95
Bước 2: ô (1,3) c=2 -> x[1,3] = 65
Bước 3: ô (4,2) c=4 -> x[4,2] = 80
Bước 4: ô (1,1) c=6 -> x[1,1] = 45
Bước 5: ô (4,4) c=6 -> x[4,4] = 20
Bước 6: ô (2,4) c=7 -> x[2,4] = 5
Bước 7: ô (3,1) c=9 -> x[3,1] = 50
Bước 8: ô (3,4) c=9 -> x[3,4] = 10

>> Phương án xuất phát:
           B1=95     B2=80     B3=65     B4=35     B5=95
A1=110  45 (c=6)   . (c=5)  65 (c=2)   . (c=7)  

(array([[45.,  0., 65.,  0.,  0.],
        [ 0.,  0.,  0.,  5., 95.],
        [50.,  0.,  0., 10.,  0.],
        [ 0., 80.,  0., 20.,  0.]]),
 <__main__.Big at 0x7f30fa339300>)

<a name="ghichu"></a>

## 7. Ghi chú phương pháp

**Quy ước dấu (theo giáo trình):** với ô chọn thì $c'_{ij}=c_{ij}+u_i+v_j=0$; phương án tối ưu khi **mọi** $c'_{ij}\ge 0$.

**Không cân bằng (mục 3.6.1):** nếu Σa < Σb (thiếu hàng) thêm *trạm phát giả* (dòng); nếu Σa > Σb (thừa hàng) thêm *trạm thu giả* (cột); cước phí trạm giả bằng 0. Giải xong **cắt bỏ dòng/cột giả**. Lượng hàng trên trạm giả = phần cung–cầu bị lệch.

**Ô cấm (mục 3.6.2):** đặt cước phí ô cấm bằng **M** (rất lớn). Trong code, mỗi giá trị là cặp $(p,q)$ nghĩa là $pM+q$, nên bảng $c'_{ij}$ hiện đúng dạng *"M−4", "−M+5"*. Khi tối ưu: nếu **mọi ô cấm đều $x=0$** → bài gốc có nghiệm; nếu **còn ô cấm là ô chọn** → **vô nghiệm**.

**Suy biến:** khi số ô chọn < m+n−1, chương trình bổ sung ô chọn 0 (tránh ô cấm). Có thể xuất hiện *bước xoay suy biến* (q=0, f không đổi) — đã ghi nhãn, là hiện tượng bình thường.

**Chỉ số ô cấm khai báo 0-based:** ô (dòng r, cột s) trong đề viết thành `(r-1, s-1)`. Ví dụ ô (1,5) → `(0,4)`.

<a name="mau"></a>

## 8. Mẫu — giải bài toán của bạn

Sửa 4 dữ liệu bên dưới rồi chạy ô code (nhớ đã chạy **Mục 1** trước đó).

In [7]:
giai_bai_toan(
    supply=[30, 50, 20],              # luong phat a_i
    demand=[20, 40, 40],              # luong thu  b_j  (co the lech tong -> tu lap tram gia)
    cost=[[2, 3, 4],
          [5, 4, 1],
          [2, 3, 3]],
    forbidden=[],                     # vi du [(0, 2)] cho o cam (1,3); de rong neu khong co
    title="Bai cua toi")

########################################################################
#  Bai cua toi
########################################################################
BƯỚC 1 - CHUẨN BỊ: đặt ô cấm = M, kiểm tra cân bằng thu-phát
Ô cấm: (không có)
Tổng phát Sum(a) = 100;  Tổng thu Sum(b) = 100
=> Cân bằng thu-phát.

Bảng cước phí (M = ô cấm; * = trạm giả):
      B1=20 B2=40 B3=40
A1=30     2     3     4
A2=50     5     4     1
A3=20     2     3     3

BƯỚC 2 - PHƯƠNG ÁN XUẤT PHÁT (ưu tiên cước phí nhỏ nhất)
Bước 1: ô (2,3) c=1 -> x[2,3] = 40
Bước 2: ô (1,1) c=2 -> x[1,1] = 20
Bước 3: ô (1,2) c=3 -> x[1,2] = 10
Bước 4: ô (3,2) c=3 -> x[3,2] = 20
Bước 5: ô (2,2) c=4 -> x[2,2] = 10

>> Phương án xuất phát:
          B1=20     B2=40     B3=40
A1=30  20 (c=2)  10 (c=3)   . (c=4)
A2=50   . (c=5)  10 (c=4)  40 (c=1)
A3=20   . (c=2)  20 (c=3)   . (c=3)

>> Số ô chọn = 5;  m + n - 1 = 5
>> f(xuất phát) = 210

VÒNG LẶP 1
BƯỚC 3 - Quy 0 ô chọn: thế vị u_i, v_j và cước phí mới c'_ij
          B1 (v=-2) B2

(array([[20., 10.,  0.],
        [ 0., 10., 40.],
        [ 0., 20.,  0.]]),
 <__main__.Big at 0x7f30fa339960>)

---
## 📝 Góp ý sau khi sử dụng

Cảm ơn bạn đã dùng thử học liệu! Xin dành khoảng **5 phút** điền **[Phiếu đánh giá mức độ đáp ứng của sản phẩm mô phỏng GeoGebra và Python](https://docs.google.com/forms/d/e/1FAIpQLScNqxvL5WIb8PxsOMvBbOrmmGTwV56bMyRV8-Nb_2Zvj225iw/viewform)** (không bắt buộc ghi họ tên). Ý kiến của bạn giúp nhóm hoàn thiện sản phẩm.

👉 **https://docs.google.com/forms/d/e/1FAIpQLScNqxvL5WIb8PxsOMvBbOrmmGTwV56bMyRV8-Nb_2Zvj225iw/viewform**

*Đề tài NCKH sinh viên TSV2026-211 – Trường Sư phạm, Đại học Cần Thơ · Kho học liệu: https://github.com/VuiPhamCTU/QHTT-GeoGebra-Python*